## The State of Tax Justice: Estimate misalignment - Mario's estimates

- Author: Mario Cuenda García, based on Alison Schultz, based on Javier Garcia Bernado's work
- Created: 4 November 2024
- Last updated: 22 September 2024

**Description**
- This notebook is the third out of three notebooks to estimate the tax losses caused by profit shifting by multinational enterprises (MNEs). The analysis used the misalignment method based on the country-by-country reports (CbCR) published by the OECD.
    - Details on the misalignment method and its background can be found here: https://www.sciencedirect.com/science/article/pii/S0305750X23003455. 
    - The working paper version is here: https://www.econstor.eu/bitstream/10419/286362/1/wp-2023-33.pdf 

- This notebook estimates profit misalignment based on different formulas. It uses the dataset **"data/final/cbcr_main.csv"** (for the estimation with imputed values) or the dataset **"data/final/cbcr_main_noimputation_allsubgroupsonly.csv"** (for the estimation without imputed values). 

**Outline**
1. Define misalignment
2. Calculate misalignment for sample with full information.
3. Calculate misalignment for samples with imputed data and aggregate results. 

**To dos before running this notebook**
- Run the notebooks 1_clean and 2_impute_missings. Note the requirements and instructions given in these notebooks.

**To dos in this notebook**

Change the formula to any required formula in each section and adapt the output name of the csvs. The formulas I have used are like follows, where sales refer to unrelated party revenues and assets to tangible assets excluding cash.

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]

- Adjust the input path in 5.1 to the bootstrapped sample you use

## 0. Load packages

In [1]:
# Packages
import pandas as pd
import numpy as np
import tjn_tools
from config import *

# Show columns and select data format
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format

[TJN TOOLS: Data processing] Module loaded.
[TJN TOOLS: Other functions] Module loaded.
[TJN TOOLS: Paths] Module loaded. Sharepoint FOUND at /Users/mariocuendagarcia/Library/CloudStorage/OneDrive-SharedLibraries-TaxJusticeNetworkLtd


## Step x. Generate the dataset with Unique ISO parents

In [2]:
# Open the original dataset
iso_parents = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Keep just the following columns: iso_parent and year
iso_parents = iso_parents[['iso_parent', 'year']]

# Keep every unique combination of iso_parent and year
iso_parents = iso_parents.drop_duplicates(subset=['iso_parent', 'year'])

# Sort by year, then iso_parent
iso_parents = iso_parents.sort_values(by=['year', 'iso_parent'])

# Filter by year
iso_parents_2016= iso_parents[iso_parents['year'] == 2016]
iso_parents_2017= iso_parents[iso_parents['year'] == 2017]
iso_parents_2018= iso_parents[iso_parents['year'] == 2018]
iso_parents_2019= iso_parents[iso_parents['year'] == 2019]
iso_parents_2020= iso_parents[iso_parents['year'] == 2020]
iso_parents_2021= iso_parents[iso_parents['year'] == 2021]

# OPTIONAL: Print the count of how many unique iso_partner values there are
print(iso_parents_2016['iso_parent'].nunique())
print(iso_parents_2017['iso_parent'].nunique())
print(iso_parents_2018['iso_parent'].nunique())
print(iso_parents_2019['iso_parent'].nunique())
print(iso_parents_2020['iso_parent'].nunique())
print(iso_parents_2021['iso_parent'].nunique())

iso_parents_2021

26
38
46
50
52
52


,iso_parent,year
0,ARE,2021
155,ARG,2021
269,AUS,2021
765,AUT,2021
797,AZE,2021
839,BEL,2021
1008,BGR,2021
1028,BHR,2021
1068,BMU,2021
1649,BRA,2021


## Step x. Generate the dataset with unique iso_partners

In [3]:
## Download the relevant dataset
iso_partners = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
iso_partners = iso_partners[~iso_partners['iso_partner'].isin(non_countries)]

# Keep just the following columns: iso_partner and year
iso_partners = iso_partners[['iso_partner', 'year']]

# Sort by year, then iso_partner
iso_partners = iso_partners.sort_values(by=['year', 'iso_partner'])

# Keep every unique combination of iso_partner and year
iso_partners = iso_partners.drop_duplicates(subset=['iso_partner', 'year'])

# Filter by year
iso_partners_2016= iso_partners[iso_partners['year'] == 2016]
iso_partners_2017= iso_partners[iso_partners['year'] == 2017]
iso_partners_2018= iso_partners[iso_partners['year'] == 2018]
iso_partners_2019= iso_partners[iso_partners['year'] == 2019]
iso_partners_2020= iso_partners[iso_partners['year'] == 2020]
iso_partners_2021= iso_partners[iso_partners['year'] == 2021]


# Optional: Print the count of how many unique iso_partner values there are 
print(iso_partners_2016['iso_partner'].nunique())
print(iso_partners_2017['iso_partner'].nunique())
print(iso_partners_2018['iso_partner'].nunique())
print(iso_partners_2019['iso_partner'].nunique())
print(iso_partners_2020['iso_partner'].nunique())
print(iso_partners_2021['iso_partner'].nunique())

183
215
213
210
212
211


## Step x. Generate the template dataset

In [4]:
# Perform a cross join to merge all values of iso_partners_ with each value of iso_parents
iso_combinations_2016 = iso_parents_2016.assign(key=1).merge(iso_partners_2016.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2017 = iso_parents_2017.assign(key=1).merge(iso_partners_2017.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2018 = iso_parents_2018.assign(key=1).merge(iso_partners_2018.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2019 = iso_parents_2019.assign(key=1).merge(iso_partners_2019.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2020 = iso_parents_2020.assign(key=1).merge(iso_partners_2020.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2021 = iso_parents_2021.assign(key=1).merge(iso_partners_2021.assign(key=1), on='key').drop('key', axis=1)

# Concatenate all the years
template_dataset = pd.concat([iso_combinations_2016, iso_combinations_2017, iso_combinations_2018, iso_combinations_2019, iso_combinations_2020, iso_combinations_2021])

# Drop year_y
template_dataset = template_dataset.drop(columns=['year_y'])
# Rename year_x to year
template_dataset = template_dataset.rename(columns={'year_x': 'year'})
# Order columns by iso_parent then iso_partner then year
template_dataset = template_dataset[['iso_parent', 'iso_partner', 'year']]
# Generate new column called cbcr_estimates
template_dataset['cbcr_estimates'] = np.nan

template_dataset

,iso_parent,iso_partner,year,cbcr_estimates
0,AUS,ABW,2016,NaN
1,AUS,AFG,2016,NaN
2,AUS,AGO,2016,NaN
3,AUS,ALB,2016,NaN
4,AUS,AND,2016,NaN
...,...,...,...,...
10967,ZAF,XKV,2021,NaN
10968,ZAF,YEM,2021,NaN
10969,ZAF,ZAF,2021,NaN
10970,ZAF,ZMB,2021,NaN


## 1. Define misalignment

In [5]:
def calculate_misalignment(cbcr_data,
                           formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",
                                         'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip'],
                           weights=[.5, 0, 0, .5, 0, 0, 0, 0],
                           profit_var='profit_loss_before_income_tax_corrected',
                           etr_max=0.15): 

    # Create variable with positive profits only for calculating shares
    cbcr_data['profit_var_pos'] = cbcr_data[profit_var]
    cbcr_data.loc[cbcr_data[profit_var] < 0, 'profit_var_pos'] = 0
    cbcr_data['share_profit'] = cbcr_data['profit_var_pos'] / cbcr_data.groupby('iso_parent')['profit_var_pos'].transform('sum')

    # Calculate weighted shares of economic activity
    actual_weights = []
    actual_variables = []
    for i, var in enumerate(formula_vars):
        if var is not None and weights[i] > 0:
            actual_variables.append(f"share_{var}")
            actual_weights.append(weights[i])
            cbcr_data.loc[cbcr_data[var] < 0, var] = 0  # Set economic activity measure to zero if negative
            cbcr_data[f"share_{var}"] = cbcr_data[var] / cbcr_data.groupby('iso_parent')[var].transform('sum')

    # Calculate the share of economic activity
    cbcr_data["share_economy_partner_of_parent"] = (cbcr_data.loc[:, actual_variables] * actual_weights).sum(1, min_count=len(actual_weights))
    # Set economic activity to 1% for those jurisdictions without economic activity but with reported profits
    cbcr_data.loc[(cbcr_data["share_economy_partner_of_parent"] == 0) & (cbcr_data[profit_var] > 0), "share_economy_partner_of_parent"] = 0.01

    # Normalize the economic activity shares to sum to 1
    cbcr_data["share_economy_partner_of_parent"] = cbcr_data["share_economy_partner_of_parent"] / cbcr_data.groupby('iso_parent')["share_economy_partner_of_parent"].transform('sum')

    # Calculate theoretical profit and misaligned profit
    cbcr_data["theoretical_profit"] = cbcr_data["share_economy_partner_of_parent"] * cbcr_data.groupby('iso_parent')[profit_var].transform('sum')
    cbcr_data["misaligned_profit"] = cbcr_data[profit_var] - cbcr_data["theoretical_profit"]

    # Set positive misaligned profits to 0 if ETR exceeds the threshold (etr_max)
    cbcr_data.loc[((cbcr_data["misaligned_profit"] > 0) & (cbcr_data["etr_average_corrected"] > etr_max)), "misaligned_profit"] = 0

    # Adjust misalignment per 'iso_parent'
    def adjust_misalignment(group):
        total_negative_misalignment = group.loc[group["misaligned_profit"] < 0, "misaligned_profit"].sum()
        total_positive_misalignment = group.loc[group["misaligned_profit"] > 0, "misaligned_profit"].sum()
        
        # Adjust negative misalignments to balance positive misalignments within each 'iso_parent'
        if total_negative_misalignment != 0:
            factor = - total_positive_misalignment / total_negative_misalignment
            group.loc[group["misaligned_profit"] < 0, "misaligned_profit"] *= factor
        
        return group

    # Apply the adjustment by grouping by 'iso_parent'
    cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)

    return cbcr_data

## 2. Calculate misalignment for sample with full information

### 2.1 Import data
- Import data without imputed values. This is only the data from the sample of reporting countries that actually is reported on a country basis, i.e. excluding aggregated country groups and data from reporting countries that do not report on a country by country basis, but just by continents.

In [6]:
cbcr_sample = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
cbcr_sample = cbcr_sample[~cbcr_sample['iso_partner'].isin(non_countries)]


# Keep cit_sample
cit_sample_2016 = cbcr_sample[cbcr_sample['year'] == 2016].copy()
# Keep iso_parent, year and cit
cit_sample_2016 = cit_sample_2016[['iso_partner', 'cit']]

# Drop duplicates
cit_sample_2016 = cit_sample_2016.drop_duplicates()

# Rename cit as cit_2016
cit_sample_2016 = cit_sample_2016.rename(columns={'cit': 'cit_2016'})

cit_sample_2016



,iso_partner,cit_2016
264,ARE,0.00
270,ARG,0.35
276,AUS,0.30
282,AUT,0.25
294,BEL,0.34
...,...,...
14778,RWA,0.30
14783,SDN,0.35
14797,SSD,0.20
14805,SWZ,0.28


### 2.2 Exclude countries that do not report truly country-by-country


- In the 2024 data, the following reporting countries do not report country-by-country. We exclude those from the "clean" analysis where we only use values that are actually in the data.
    - Austria: Only continents in all years
    - Czechia: Only Czechia versus rest of the world from 2019 to 2021
    - Finland: Only Finland and rest of the world between 2016 and 2018 and Finland and continents between 2019 and 2021
    - Greece: Only Greece and continents between 2017 and 2019
    - Hungary: Only Hungary versus rest of the world between 2018 and 2021
    - Isle of Man: Only continents between 2017 and 2020
    - Ireland: Only Ireland versus rest of the world in all years
    - Korea: Only Korea and rest of the world betweem 2016 and 2018 and Korea and continents between 2019 and 2021
    - Macau: Only Macau versus rest of the world between 2019 and 2021
    - Mauritius: Only Mauritius and continents between 2019 and 2021
    - Morocco: Only Morocco and continents in 2021
    - Netherlands: Only Netherlands versus rest of the world between 2016 and 2017
    - Norway: Only Norway and continents 2016 and 2017
    - New Zealand: Only New Zealand versus rest of the world between 2018 and 2021
    - Poland: Only Poland and continents 2019 to 2021
    - Sweden: Only Sweden and continents in all years
    - United Kingdom: Only UK and continents between 2017 and 2021

In [7]:
# Define the conditions for exclusion
exclusion_conditions = [
    ('AUT', 2016, 2021),                # Austria: all years
    ('CZE', 2019, 2021),                # Czechia: from 2019 to 2021
    ('FIN', 2016, 2021),                # Finland: all years
    ('GRC', 2017, 2019),                # Greece: between 2017 and 2019
    ('HUN', 2018, 2021),                # Hungary: between 2018 and 2021
    ('IMN', 2017, 2020),                # Isle of Man: between 2017 and 2020
    ('IRL', 2016, 2021),                # Ireland: all years
    ('KOR', 2016, 2021),                # Korea: all years
    ('MAC', 2019, 2021),                # Macau: between 2019 and 2021
    ('MUS', 2019, 2021),                # Mauritius: between 2019 and 2021
    ('MAR', 2021, 2021),                # Morocco: 2021
    ('NLD', 2016, 2017),                # Netherlands: between 2016 and 2017
    ('NOR', 2016, 2017),                # Norway: 2016 and 2017
    ('NZL', 2018, 2021),                # New Zealand: between 2018 and 2021
    ('POL', 2019, 2021),                # Poland: 2019 to 2021
    ('SWE', 2016, 2021),                # Sweden: all years
    ('GBR', 2017, 2021)                 # United Kingdom: between 2017 and 2021
]

# Iterate through the exclusion conditions
for iso_parent, start_year, end_year in exclusion_conditions:
    cbcr_sample = cbcr_sample[~((cbcr_sample['iso_parent'] == iso_parent) & 
                                     (cbcr_sample['year'].between(start_year, end_year)))]

In [8]:
cbcr_sample

,iso_parent,parent_jurisdiction,iso_partner,partner_jurisdiction,year,unrelated_party_revenues,profit_loss_before_income_tax,adjusted_profit_loss_before_income_tax,income_tax_paid_on_cash_basis,income_tax_accrued_current_year,n_employees,tangible_assets_except_cash,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,n_cbcr,n_cbcr_groups,n_entities,profit_loss_before_income_tax_corrected,ln_profit_loss_before_income_tax_corrected,ln_unrelated_party_revenues,ln_n_employees,ln_tangible_assets_except_cash,ln_stated_capital,ln_total_revenues,ln_related_party_revenues,ln_holding_or_managing_ip,etr_domestic,etr_domestic_corrected,etr_foreign,etr_foreign_corrected,etr_average,etr_average_corrected,cit,gdp_current_usd,population,gdp,wage_monthly,payroll,ln_wage_monthly,ln_gdp_current_usd,ln_population,gvt_health_expenditure,ln_gvt_health_expenditure,tax_revenue_pct_gdp,tax_revenue_current_usd,cthi_2021_share,cthi_2021_score,region_tjn,ukt,gbr_oct,nld_oct,oecd_oct,oecd,eu
0,ARE,United Arab Emirates,AFG,Afghanistan,2021,"143,907,630.00","-46,006,802.00",NaN,"42,927,479.00",0.00,"3,564.00","78,832,974.00","48,663,869.00","148,803,203.00","4,895,573.00",NaN,5.00,5.00,18.00,"-46,006,802.00",0.00,18.78,8.18,18.18,17.70,18.82,15.40,NaN,NaN,NaN,0.07,0.07,0.07,0.07,0.20,"14,266,499,429.87","40,099,462.00",NaN,171.87,"7,350,707.23",5.15,23.38,17.51,"107,697,735.61",18.49,NaN,NaN,NaN,NaN,Asia,0.00,0.00,0.00,0.00,0.00,0.00
1,ARE,United Arab Emirates,AGO,Angola,2021,"171,764,219.00","17,925,795.00",NaN,"13,390,865.00","9,822,146.00","3,249.00","22,250,946.00","10,236,316.00","171,764,219.00",0.00,NaN,4.00,4.00,10.00,"17,925,795.00",16.70,18.96,8.09,16.92,16.14,18.96,0.00,NaN,NaN,NaN,0.36,0.36,0.36,0.36,0.25,"66,505,129,987.72","34,503,774.00",NaN,143.61,"5,598,910.73",4.97,24.92,17.36,"1,279,772,420.68",20.97,NaN,NaN,NaN,NaN,Africa,0.00,0.00,0.00,0.00,0.00,0.00
2,ARE,United Arab Emirates,ARE,United Arab Emirates,2021,"201,000,000,000.00","31,538,024,664.00",NaN,"136,782,996.00","61,896,681.00","581,795.00","342,000,000,000.00","443,000,000,000.00","257,000,000,000.00","56,775,290,388.00",96.00,59.00,59.00,"4,462.00","31,538,024,664.00",24.17,26.03,13.27,26.56,26.82,26.27,24.76,4.57,0.03,0.03,0.34,0.35,0.20,0.20,0.00,"415,178,792,756.98","9,365,145.00",NaN,"3,161.18","22,069,905,037.20",8.06,26.75,16.05,"14,119,521,288.48",23.37,0.54,"2,222,218,203.26",0.04,98.33,Asia,0.00,0.00,0.00,0.00,0.00,0.00
3,ARE,United Arab Emirates,ARG,Argentina,2021,"297,786,434.00","35,504,518.00",NaN,"7,470,407.00","13,543,107.00","2,007.00","150,688,454.00","92,323,233.00","297,786,434.00",0.00,NaN,6.00,6.00,16.00,"35,504,518.00",17.39,19.51,7.60,18.83,18.34,19.51,0.00,NaN,0.17,0.19,0.28,0.29,0.25,0.26,0.30,"487,902,572,164.35","45,808,747.00",NaN,544.82,"13,121,420.80",6.30,26.91,17.64,"29,899,372,168.89",24.12,11.47,"55,955,195,677.26",0.00,32.13,Latin America,0.00,0.00,0.00,0.00,0.00,0.00
4,ARE,United Arab Emirates,ARM,Armenia,2021,"12,587,092.00","-321,915.00",NaN,0.00,0.00,327.00,"4,594,136.00","29,657,109.00","12,587,092.00",0.00,NaN,2.00,2.00,3.00,"-321,915.00",0.00,16.35,5.79,15.34,17.21,16.35,0.00,NaN,NaN,NaN,0.44,0.45,0.44,0.45,0.18,"13,878,908,628.94","2,790,974.00",NaN,259.16,"1,016,959.54",5.56,23.35,14.84,"303,765,192.71",19.53,22.00,"3,053,499,345.31",NaN,NaN,Asia,0.00,0.00,0.00,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14879,ZAF,South Africa,ZWE,Zimbabwe,2017,"954,173,462.00","212,146,066.00",NaN,"71,754,179.00","83,024,004.00","19,560.00","1,834,868,152.00","279,445,783.00","1,483,815,457.00","529,641,995.00",NaN,21.00,21.00,45.00,"212,146,066.00",19.17,20.68,9.88,21.33,19.45,21.12,20.09,NaN,NaN,NaN,0.18,0.18,0.18,0.18,0.26,"17,584,890,936.65","14,751,101.00",NaN,255.97,"60,080,926.32",5.55,23.59,16.51,"341,835,130.00",19.65,15.87,"

### 2.3 Calculate misalignment for sample countries with full information

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]


In [9]:
misalignment_2021 = cbcr_sample[cbcr_sample['year'] == 2021].copy()

# Merge cit_sample_2016 with misalignment_2021
misalignment_2021 = misalignment_2021.merge(cit_sample_2016, on='iso_partner', how='left')

# Keep iso_partner, cit and cit_2016
comparison_cits = misalignment_2021[['iso_partner', 'cit', 'oecd', 'cit_2016']]

# Drop column cit
misalignment_2021 = misalignment_2021.drop(columns=['cit'])
# Rename cit_2016 as cit
misalignment_2021 = misalignment_2021.rename(columns={'cit_2016': 'cit'})

misalignment_2021 = calculate_misalignment(misalignment_2021, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Keep only the first occurrence of these unique variables for each 'iso_partner'
unique_columns = misalignment_2021.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                               'etr_average_corrected', 'cit',
                                                                               'tax_revenue_current_usd', 
                                                                               'gvt_health_expenditure', 'region_tjn', 
                                                                               'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

# Keep iso_parent, iso_partner, year, misaligned_profit, theoretical_profit, profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
misalignment_2021 = misalignment_2021[['iso_parent', 'iso_partner', 'oecd', 'year', 'misaligned_profit', 'theoretical_profit', 'profit_loss_before_income_tax_corrected', 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']]

# Show if iso_partner = USA
misalignment_2021[misalignment_2021['iso_partner'] == 'USA']

/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_23015/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


,iso_parent,iso_partner,oecd,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip
140,ARE,USA,1.00,2021,"-1,211,834,974.56","1,676,278,193.98","-566,015,448.00","28,384.00","22,510,101,353.00","11,019,650,222.00","1,566,840,057.22","33,844,332,143.00","25,838,686,878.00","3,328,585,524.00",19.00
165,ARG,USA,1.00,2021,0.00,"9,480,880.09","67,155,125.00",114.00,"113,406,031.00","103,940,740.00","6,292,973.74","1,762,118,722.00","132,596,801.00","19,190,770.00",2.00
245,AUS,USA,1.00,2021,0.00,"7,200,422,731.80","8,095,396,499.00","89,401.00","57,209,834,217.00","37,039,556,556.00","4,935,071,447.12","174,000,000,000.00","73,738,510,494.00","16,528,676,277.00",27.00
285,AZE,USA,1.00,2021,"-8,636,245.39","10,751,113.69","-5,934,134.00",21.00,"748,376,581.00","7,712,734.00","1,159,232.00","20,001,000.00","751,937,979.00","3,561,398.00",NaN
314,BEL,USA,1.00,2021,"-553,930,903.11","3,029,537,331.17","2,376,700,000.00","58,400.00","43,812,500,000.00","14,989,500,000.00","3,223,769,001.60","380,000,000,000.00","54,828,300,000.00","11,015,800,000.00",25.00
353,BHR,USA,1.00,2021,0.00,"5,748,649.03","28,628,903.00",46.00,"31,352,002.00","494,407.00","2,539,270.10","1,253,597.00","491,165,142.00","459,813,140.00",NaN
441,BMU,USA,1.00,2021,"-2,631,136,978.26","11,377,613,393.13","8,368,805,189.00","95,211.00","87,810,248,119.00","43,717,634,640.00","5,255,792,301.56","75,203,353,654.00","109,000,000,000.00","21,625,977,894.00",16.00
480,BRA,USA,1.00,2021,"-1,561,921,714.72","12,227,561,481.77","10,586,870,772.00","121,286.00","68,461,937,528.00","17,087,060,697.00","6,695,172,039.86","47,722,813,871.00","79,304,813,607.00","10,842,876,078.00",20.00
493,CAN,USA,1.00,2021,"-62,835,521,174.62","120,748,616,174.62","57,913,095,000.00","867,200.00","448,000,000,000.00","440,000,000,000.00","47,870,761,612.80","1,150,000,000,000.00","561,000,000,000.00","112,000,000,000.00",NaN
623,CHE,USA,1.00,2021,"-2,611,278,125.48","17,075,485,567.55","13,563,295,251.00","303,765.00","267,000,000,000.00","91,543,939,101.00","16,768,290,937.86","398,000,000,000.00","334,000,000,000.00","67,308,695,757.00",64.00


In [10]:
# Calculate the total sum of profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
total_profit_loss_before_income_tax_corrected = misalignment_2021['profit_loss_before_income_tax_corrected'].sum()
total_n_employees = misalignment_2021['n_employees'].sum()
total_unrelated_party_revenues = misalignment_2021['unrelated_party_revenues'].sum()
total_tangible_assets_except_cash = misalignment_2021['tangible_assets_except_cash'].sum()
total_payroll = misalignment_2021['payroll'].sum()
total_stated_capital = misalignment_2021['stated_capital'].sum()
total_total_revenues = misalignment_2021['total_revenues'].sum()
total_related_party_revenues = misalignment_2021['related_party_revenues'].sum()
total_holding_or_managing_ip = misalignment_2021['holding_or_managing_ip'].sum()

misalignment_2021['total_profit_loss_before_income_tax_corrected'] = total_profit_loss_before_income_tax_corrected
misalignment_2021['total_n_employees'] = total_n_employees
misalignment_2021['total_unrelated_party_revenues'] = total_unrelated_party_revenues
misalignment_2021['total_tangible_assets_except_cash'] = total_tangible_assets_except_cash
misalignment_2021['total_payroll'] = total_payroll
misalignment_2021['total_stated_capital'] = total_stated_capital
misalignment_2021['total_total_revenues'] = total_total_revenues
misalignment_2021['total_related_party_revenues'] = total_related_party_revenues
misalignment_2021['total_holding_or_managing_ip'] = total_holding_or_managing_ip

# Group by iso_partner and calculate the total sum of profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
total_profit_loss_by_partner = misalignment_2021.groupby('iso_partner')['profit_loss_before_income_tax_corrected'].sum().reset_index()
total_profit_loss_by_partner = total_profit_loss_by_partner.rename(columns={'profit_loss_before_income_tax_corrected': 'total_profit_loss_by_partner'})

total_n_employees_by_partner = misalignment_2021.groupby('iso_partner')['n_employees'].sum().reset_index()
total_n_employees_by_partner = total_n_employees_by_partner.rename(columns={'n_employees': 'total_n_employees_by_partner'})

total_unrelated_party_revenues_by_partner = misalignment_2021.groupby('iso_partner')['unrelated_party_revenues'].sum().reset_index()
total_unrelated_party_revenues_by_partner = total_unrelated_party_revenues_by_partner.rename(columns={'unrelated_party_revenues': 'total_unrelated_party_revenues_by_partner'})

total_tangible_assets_except_cash_by_partner = misalignment_2021.groupby('iso_partner')['tangible_assets_except_cash'].sum().reset_index()
total_tangible_assets_except_cash_by_partner = total_tangible_assets_except_cash_by_partner.rename(columns={'tangible_assets_except_cash': 'total_tangible_assets_except_cash_by_partner'})

total_payroll_by_partner = misalignment_2021.groupby('iso_partner')['payroll'].sum().reset_index()
total_payroll_by_partner = total_payroll_by_partner.rename(columns={'payroll': 'total_payroll_by_partner'})

total_stated_capital_by_partner = misalignment_2021.groupby('iso_partner')['stated_capital'].sum().reset_index()
total_stated_capital_by_partner = total_stated_capital_by_partner.rename(columns={'stated_capital': 'total_stated_capital_by_partner'})

total_total_revenues_by_partner = misalignment_2021.groupby('iso_partner')['total_revenues'].sum().reset_index()
total_total_revenues_by_partner = total_total_revenues_by_partner.rename(columns={'total_revenues': 'total_total_revenues_by_partner'})

total_related_party_revenues_by_partner = misalignment_2021.groupby('iso_partner')['related_party_revenues'].sum().reset_index()
total_related_party_revenues_by_partner = total_related_party_revenues_by_partner.rename(columns={'related_party_revenues': 'total_related_party_revenues_by_partner'})

total_holding_or_managing_ip_by_partner = misalignment_2021.groupby('iso_partner')['holding_or_managing_ip'].sum().reset_index()
total_holding_or_managing_ip_by_partner = total_holding_or_managing_ip_by_partner.rename(columns={'holding_or_managing_ip': 'total_holding_or_managing_ip_by_partner'})

# Merge the total profit loss by partner back into the misalignment_2021 dataframe
misalignment_2021 = misalignment_2021.merge(total_profit_loss_by_partner, on='iso_partner', how='left')
misalignment_2021 = misalignment_2021.merge(total_n_employees_by_partner, on='iso_partner', how='left')
misalignment_2021 = misalignment_2021.merge(total_unrelated_party_revenues_by_partner, on='iso_partner', how='left')
misalignment_2021 = misalignment_2021.merge(total_tangible_assets_except_cash_by_partner, on='iso_partner', how='left')
misalignment_2021 = misalignment_2021.merge(total_payroll_by_partner, on='iso_partner', how='left')
misalignment_2021 = misalignment_2021.merge(total_stated_capital_by_partner, on='iso_partner', how='left')
misalignment_2021 = misalignment_2021.merge(total_total_revenues_by_partner, on='iso_partner', how='left')
misalignment_2021 = misalignment_2021.merge(total_related_party_revenues_by_partner, on='iso_partner', how='left')
misalignment_2021 = misalignment_2021.merge(total_holding_or_managing_ip_by_partner, on='iso_partner', how='left')

misalignment_2021[misalignment_2021['iso_partner'] == 'USA']

,iso_parent,iso_partner,oecd,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,total_profit_loss_before_income_tax_corrected,total_n_employees,total_unrelated_party_revenues,total_tangible_assets_except_cash,total_payroll,total_stated_capital,total_total_revenues,total_related_party_revenues,total_holding_or_managing_ip,total_profit_loss_by_partner,total_n_employees_by_partner,total_unrelated_party_revenues_by_partner,total_tangible_assets_except_cash_by_partner,total_payroll_by_partner,total_stated_capital_by_partner,total_total_revenues_by_partner,total_related_party_revenues_by_partner,total_holding_or_managing_ip_by_partner
140,ARE,USA,1.00,2021,"-1,211,834,974.56","1,676,278,193.98","-566,015,448.00","28,384.00","22,510,101,353.00","11,019,650,222.00","1,566,840,057.22","33,844,332,143.00","25,838,686,878.00","3,328,585,524.00",19.00,"7,762,091,416,141.00","157,545,260.00","61,215,345,239,302.00","42,955,732,312,253.00","4,519,767,206,421.01","107,533,344,406,981.00","86,567,376,841,818.00","25,321,553,863,775.00","22,468.00","2,091,232,518,307.00","30,088,947.00","16,720,675,917,579.00","9,446,740,674,823.00","1,660,955,729,955.23","22,120,338,504,730.00","22,139,101,840,847.00","5,389,605,935,829.00","1,503.00"
165,ARG,USA,1.00,2021,0.00,"9,480,880.09","67,155,125.00",114.00,"113,406,031.00","103,940,740.00","6,292,973.74","1,762,118,722.00","132,596,801.00","19,190,770.00",2.00,"7,762,091,416,141.00","157,545,260.00","61,215,345,239,302.00","42,955,732,312,253.00","4,519,767,206,421.01","107,533,344,406,981.00","86,567,376,841,818.00","25,321,553,863,775.00","22,468.00","2,091,232,518,307.00","30,088,947.00","16,720,675,917,579.00","9,446,740,674,823.00","1,660,955,729,955.23","22,120,338,504,730.00","22,139,101,840,847.00","5,389,605,935,829.00","1,503.00"
245,AUS,USA,1.00,2021,0.00,"7,200,422,731.80","8,095,396,499.00","89,401.00","57,209,834,217.00","37,039,556,556.00","4,935,071,447.12","174,000,000,000.00","73,738,510,494.00","16,528,676,277.00",27.00,"7,762,091,416,141.00","157,545,260.00","61,215,345,239,302.00","42,955,732,312,253.00","4,519,767,206,421.01","107,533,344,406,981.00","86,567,376,841,818.00","25,321,553,863,775.00","22,468.00","2,091,232,518,307.00","30,088,947.00","16,720,675,917,579.00","9,446,740,674,823.00","1,660,955,729,955.23","22,120,338,504,730.00","22,139,101,840,847.00","5,389,605,935,829.00","1,503.00"
285,AZE,USA,1.00,2021,"-8,636,245.39","10,751,113.69","-5,934,134.00",21.00,"748,376,581.00","7,712,734.00","1,159,232.00","20,001,000.00","751,937,979.00","3,561,398.00",NaN,"7,762,091,416,141.00","157,545,260.00","61,215,345,239,302.00","42,955,732,312,253.00","4,519,767,206,421.01","107,533,344,406,981.00","86,567,376,841,818.00","25,321,553,863,775.00","22,468.00","2,091,232,518,307.00","30,088,947.00","16,720,675,917,579.00","9,446,740,674,823.00","1,660,955,729,955.23","22,120,338,504,730.00","22,139,101,840,847.00","5,389,605,935,829.00","1,503.00"
314,BEL,USA,1.00,2021,"-553,930,903.11","3,029,537,331.17","2,376,700,000.00","58,400.00","43,812,500,000.00","14,989,500,000.00","3,223,769,001.60","380,000,000,000.00","54,828,300,000.00","11,015,800,000.00",25.00,"7,762,091,416,141.00","157,545,260.00","61,215,345,239,302.00","42,955,732,312,253.00","4,519,767,206,421.01","107,533,344,406,981.00","86,567,376,841,818.00","25,321,553,863,775.00","22,468.00","2,091,232,518,307.00","30,088,947.00","16,720,675,917,579.00","9,446,740,674,823.00","1,660,955,729,955.23","22,120,338,504,730.00","22,139,101,840,847.00","5,389,605,935,829.00","1,503.00"
353,BHR,USA,1.00,2021,0.00,"5,748,649.03","28,628,903.00",46.00,"31,352,002.00","494,407.00","2,539,270.10","1,253,597.00","491,165,142.00","459,813,140.00",NaN,"7,762,091,416,141.00","157,545,260.00","61,215,345,239,302.00","42,955,732,312,253.00","4,519,767,206,421.

In [11]:
# Final Misalignment
final_misalignment_2021 = misalignment_2021

# Calculate the shares for all variables
final_misalignment_2021['share_reported_total_profit_loss_by_partner'] = misalignment_2021['total_profit_loss_by_partner'] / misalignment_2021['total_profit_loss_before_income_tax_corrected']
final_misalignment_2021['share_reported_total_n_employees_by_partner'] = misalignment_2021['total_n_employees_by_partner'] / misalignment_2021['total_n_employees']
final_misalignment_2021['share_reported_total_unrelated_party_revenues_by_partner'] = misalignment_2021['total_unrelated_party_revenues_by_partner'] / misalignment_2021['total_unrelated_party_revenues']
final_misalignment_2021['share_reported_total_tangible_assets_except_cash_by_partner'] = misalignment_2021['total_tangible_assets_except_cash_by_partner'] / misalignment_2021['total_tangible_assets_except_cash']
final_misalignment_2021['share_reported_total_payroll_by_partner'] = misalignment_2021['total_payroll_by_partner'] / misalignment_2021['total_payroll']
final_misalignment_2021['share_reported_total_stated_capital_by_partner'] = misalignment_2021['total_stated_capital_by_partner'] / misalignment_2021['total_stated_capital']
final_misalignment_2021['share_reported_total_total_revenues_by_partner'] = misalignment_2021['total_total_revenues_by_partner'] / misalignment_2021['total_total_revenues']
final_misalignment_2021['share_reported_total_related_party_revenues_by_partner'] = misalignment_2021['total_related_party_revenues_by_partner'] / misalignment_2021['total_related_party_revenues']
final_misalignment_2021['share_reported_total_holding_or_managing_ip_by_partner'] = misalignment_2021['total_holding_or_managing_ip_by_partner'] / misalignment_2021['total_holding_or_managing_ip']

# Give me column 'share reported' with 2 decimals
pd.options.display.float_format = '{:,.2f}'.format

final_misalignment_2021[final_misalignment_2021['iso_partner'] == 'USA']

,iso_parent,iso_partner,oecd,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,total_profit_loss_before_income_tax_corrected,total_n_employees,total_unrelated_party_revenues,total_tangible_assets_except_cash,total_payroll,total_stated_capital,total_total_revenues,total_related_party_revenues,total_holding_or_managing_ip,total_profit_loss_by_partner,total_n_employees_by_partner,total_unrelated_party_revenues_by_partner,total_tangible_assets_except_cash_by_partner,total_payroll_by_partner,total_stated_capital_by_partner,total_total_revenues_by_partner,total_related_party_revenues_by_partner,total_holding_or_managing_ip_by_partner,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
140,ARE,USA,1.00,2021,"-1,211,834,974.56","1,676,278,193.98","-566,015,448.00","28,384.00","22,510,101,353.00","11,019,650,222.00","1,566,840,057.22","33,844,332,143.00","25,838,686,878.00","3,328,585,524.00",19.00,"7,762,091,416,141.00","157,545,260.00","61,215,345,239,302.00","42,955,732,312,253.00","4,519,767,206,421.01","107,533,344,406,981.00","86,567,376,841,818.00","25,321,553,863,775.00","22,468.00","2,091,232,518,307.00","30,088,947.00","16,720,675,917,579.00","9,446,740,674,823.00","1,660,955,729,955.23","22,120,338,504,730.00","22,139,101,840,847.00","5,389,605,935,829.00","1,503.00",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
165,ARG,USA,1.00,2021,0.00,"9,480,880.09","67,155,125.00",114.00,"113,406,031.00","103,940,740.00","6,292,973.74","1,762,118,722.00","132,596,801.00","19,190,770.00",2.00,"7,762,091,416,141.00","157,545,260.00","61,215,345,239,302.00","42,955,732,312,253.00","4,519,767,206,421.01","107,533,344,406,981.00","86,567,376,841,818.00","25,321,553,863,775.00","22,468.00","2,091,232,518,307.00","30,088,947.00","16,720,675,917,579.00","9,446,740,674,823.00","1,660,955,729,955.23","22,120,338,504,730.00","22,139,101,840,847.00","5,389,605,935,829.00","1,503.00",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
245,AUS,USA,1.00,2021,0.00,"7,200,422,731.80","8,095,396,499.00","89,401.00","57,209,834,217.00","37,039,556,556.00","4,935,071,447.12","174,000,000,000.00","73,738,510,494.00","16,528,676,277.00",27.00,"7,762,091,416,141.00","157,545,260.00","61,215,345,239,302.00","42,955,732,312,253.00","4,519,767,206,421.01","107,533,344,406,981.00","86,567,376,841,818.00","25,321,553,863,775.00","22,468.00","2,091,232,518,307.00","30,088,947.00","16,720,675,917,579.00","9,446,740,674,823.00","1,660,955,729,955.23","22,120,338,504,730.00","22,139,101,840,847.00","5,389,605,935,829.00","1,503.00",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
285,AZE,USA,1.00,2021,"-8,636,245.39","10,751,113.69","-5,934,134.00",21.00,"748,376,581.00","7,712,734.00","1,159,232.00","20,001,000.00","751,937,979.00","3,561,398.00",NaN,"7,762,091,416,141.00","157,545,260.00","61,215,345,239,302.00","42,955,732,312,253.00","4,519,767,206,421.01","107,533,344,406,981.00","86,567,376,841,818.00","25,321,553,863,775.00","22,468.00","2,091,232,518,307.00","30,088,947.00","16,720,675,917,579.00","9,446,740,674,823.00","1,660,955,729,955.23","22,120,338,504,730.00","22,139,101,840,847.00","5,389,605,935,829.00","1,503.00",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
314,BEL,USA,1.00,2021,"-553,930,903.11","3,029,537,331.17","2,376,700,000.00","58,400.00","43,812,500,000.00","14,989,500,000.00","3,223,769,001.60","380,000,000,000.00","54,828,300,000.00","11,015,800,000.00",25.00,"7,762,091,416,141.00","15

In [12]:
# Keep iso_partner and share_reported_total_profit_loss_by_partner	share_reported_total_n_employees_by_partner	share_reported_total_unrelated_party_revenues_by_partner	share_reported_total_tangible_assets_except_cash_by_partner	share_reported_total_payroll_by_partner	share_reported_total_stated_capital_by_partner	share_reported_total_total_revenues_by_partner	share_reported_total_related_party_revenues_by_partner	share_reported_total_holding_or_managing_ip_by_partner
shares_reported_2021 = final_misalignment_2021[['iso_partner', 'share_reported_total_profit_loss_by_partner', 'share_reported_total_n_employees_by_partner', 'share_reported_total_unrelated_party_revenues_by_partner', 'share_reported_total_tangible_assets_except_cash_by_partner', 'share_reported_total_payroll_by_partner', 'share_reported_total_stated_capital_by_partner', 'share_reported_total_total_revenues_by_partner', 'share_reported_total_related_party_revenues_by_partner', 'share_reported_total_holding_or_managing_ip_by_partner']]

# Drop duplicates
shares_reported_2021 = shares_reported_2021.drop_duplicates()

shares_reported_2021[shares_reported_2021['iso_partner'] == 'USA']

,iso_partner,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
140,USA,0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07


In [13]:
excluded_2021 = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Keep if year == 2021 and iso_parent == 'AUT', 'FIN', 'IRL0', 'KOR', 'NLD', 'NOR' 'SWE'
excluded_2021 = excluded_2021[excluded_2021['year'] == 2021]
excluded_2021 = excluded_2021[excluded_2021['iso_parent'].isin(['AUT', 'CZE', 'FIN', 'HUN', 'IRL', 'KOR', 'MAC', 'MUS', 'MAR' 'NZL', 'POL', 'SWE', 'GBR'])]

# Sum by iso_parent: 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip' and 'profit_loss_before_income_tax_corrected'
excluded_2021 = excluded_2021.groupby('iso_parent').agg({'n_employees': 'sum', 'unrelated_party_revenues': 'sum', 'tangible_assets_except_cash': 'sum', 'payroll': 'sum', 'stated_capital': 'sum', 'total_revenues': 'sum', 'related_party_revenues': 'sum', 'holding_or_managing_ip': 'sum', 'profit_loss_before_income_tax_corrected': 'sum'}).reset_index()

# Merge iso_combinations_2021 with excluded_2021. 
excluded_jurisdictions_2021 = pd.merge(iso_combinations_2021, excluded_2021, on='iso_parent', how='left')

# Drop year_y
excluded_jurisdictions_2021 = excluded_jurisdictions_2021.drop(columns=['year_y'])
# Rename year_x to year
excluded_jurisdictions_2021 = excluded_jurisdictions_2021.rename(columns={'year_x': 'year'})

# Keep if iso_parent == 'AUT', 'FIN', 'IRL0', 'KOR', 'NLD', 'NOR' 'SWE'
excluded_jurisdictions_2021 = excluded_jurisdictions_2021[excluded_jurisdictions_2021['iso_parent'].isin(['AUT', 'CZE', 'FIN', 'HUN', 'IRL', 'KOR', 'MAC', 'MUS', 'MAR' 'NZL', 'POL', 'SWE', 'GBR'])]

excluded_jurisdictions_2021[excluded_jurisdictions_2021['iso_partner'] == 'USA']


,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected
830,AUT,2021,USA,"2,069,263.00","726,297,147,483.00","401,888,660,314.00","22,529,811,546.62","270,964,405,217.00","951,640,549,767.00","224,852,493,775.00",580.00,"63,174,139,580.00"
3362,CZE,2021,USA,"444,663.00","238,000,000,000.00","53,690,818,054.00","2,606,664,163.22","30,586,349,491.00","365,000,000,000.00","125,750,238,269.00",0.00,"5,814,794,701.00"
4206,FIN,2021,USA,"985,297.00","817,761,022,659.00","184,844,536,829.00","10,156,173,245.81","612,898,954,550.00","1,130,883,735,340.00","311,960,968,624.00",217.00,"42,314,748,301.00"
4628,GBR,2021,USA,"10,746,171.00","4,880,906,637,939.00","3,615,569,195,881.00","112,780,002,636.05","13,839,749,415,401.00","7,180,000,000,000.00","2,297,536,542,295.00","3,590.00","604,216,148,740.00"
5261,HUN,2021,USA,"134,329.00","58,109,599,064.00","19,550,252,423.00","959,278,134.14","15,860,548,301.00","74,640,793,509.00","16,531,194,445.00",0.00,"6,868,840,612.00"
5894,IRL,2021,USA,"1,915,063.00","427,571,530,900.00","217,000,000,000.00","7,126,386,684.12","2,369,000,000,000.00","697,000,000,000.00","269,410,895,368.00",376.00,"36,519,731,063.00"
6527,KOR,2021,USA,"6,345,530.00","3,019,577,465,309.00","2,541,934,539,062.00","77,256,150,290.88","1,089,902,961,198.00","4,316,949,217,844.00","1,298,930,513,223.00","1,516.00","276,597,253,936.00"
7793,MUS,2021,USA,"101,419.00","170,242,775,061.00","52,766,072,303.00","136,773,695.63","62,329,244,580.00","187,866,781,453.00","17,624,006,392.00",75.00,"7,439,064,276.00"
10325,SWE,2021,USA,"2,924,649.00","868,921,664,525.00","409,722,741,330.00","22,514,030,335.37","336,932,617,882.00","1,252,041,349,614.00","382,298,181,710.00","1,274.00","129,576,740,443.00"


In [14]:
# Merge with share_reported_2021
excluded_jurisdictions_share_reported_2021 = pd.merge(excluded_jurisdictions_2021, shares_reported_2021, on='iso_partner', how='left')

# Multiply 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip' and 'profit_loss_before_income_tax_corrected' by share_reported
excluded_jurisdictions_share_reported_2021['n_employees'] = excluded_jurisdictions_share_reported_2021['n_employees'] * excluded_jurisdictions_share_reported_2021['share_reported_total_n_employees_by_partner']
excluded_jurisdictions_share_reported_2021['unrelated_party_revenues'] = excluded_jurisdictions_share_reported_2021['unrelated_party_revenues'] * excluded_jurisdictions_share_reported_2021['share_reported_total_unrelated_party_revenues_by_partner']
excluded_jurisdictions_share_reported_2021['tangible_assets_except_cash'] = excluded_jurisdictions_share_reported_2021['tangible_assets_except_cash'] * excluded_jurisdictions_share_reported_2021['share_reported_total_tangible_assets_except_cash_by_partner']
excluded_jurisdictions_share_reported_2021['payroll'] = excluded_jurisdictions_share_reported_2021['payroll'] * excluded_jurisdictions_share_reported_2021['share_reported_total_payroll_by_partner']
excluded_jurisdictions_share_reported_2021['stated_capital'] = excluded_jurisdictions_share_reported_2021['stated_capital'] * excluded_jurisdictions_share_reported_2021['share_reported_total_stated_capital_by_partner']
excluded_jurisdictions_share_reported_2021['total_revenues'] = excluded_jurisdictions_share_reported_2021['total_revenues'] * excluded_jurisdictions_share_reported_2021['share_reported_total_total_revenues_by_partner']
excluded_jurisdictions_share_reported_2021['related_party_revenues'] = excluded_jurisdictions_share_reported_2021['related_party_revenues'] * excluded_jurisdictions_share_reported_2021['share_reported_total_related_party_revenues_by_partner']
excluded_jurisdictions_share_reported_2021['holding_or_managing_ip'] = excluded_jurisdictions_share_reported_2021['holding_or_managing_ip'] * excluded_jurisdictions_share_reported_2021['share_reported_total_holding_or_managing_ip_by_partner']
excluded_jurisdictions_share_reported_2021['profit_loss_before_income_tax_corrected'] = excluded_jurisdictions_share_reported_2021['profit_loss_before_income_tax_corrected'] * excluded_jurisdictions_share_reported_2021['share_reported_total_profit_loss_by_partner']

excluded_jurisdictions_share_reported_2021[excluded_jurisdictions_share_reported_2021['iso_partner'] == 'USA']

,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected,share_reported_total_profit_loss_by_partner,share_reported_total_n_employees_by_partner,share_reported_total_unrelated_party_revenues_by_partner,share_reported_total_tangible_assets_except_cash_by_partner,share_reported_total_payroll_by_partner,share_reported_total_stated_capital_by_partner,share_reported_total_total_revenues_by_partner,share_reported_total_related_party_revenues_by_partner,share_reported_total_holding_or_managing_ip_by_partner
197,AUT,2021,USA,"395,200.37","198,384,558,241.92","88,382,568,513.57","8,279,413,048.09","55,739,216,511.75","243,376,521,454.14","47,859,082,489.78",38.80,"17,020,131,292.33",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
408,CZE,2021,USA,"84,924.43","65,008,550,598.34","11,807,579,744.85","957,915,215.59","6,291,819,603.45","93,346,621,634.09","26,765,507,134.86",0.00,"1,566,599,401.38",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
619,FIN,2021,USA,"188,177.98","223,367,474,028.89","40,650,649,182.00","3,732,261,724.25","126,077,473,165.25","289,216,920,972.42","66,399,822,747.38",14.52,"11,400,275,120.06",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
830,GBR,2021,USA,"2,052,368.76","1,333,196,076,210.98","795,128,909,387.10","41,445,185,791.12","2,846,930,350,558.64","1,836,243,132,418.60","489,022,776,909.91",240.15,"162,785,567,779.30",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
1041,HUN,2021,USA,"25,654.97","15,872,356,348.74","4,299,453,293.56","352,522,251.87","3,262,622,391.44","19,088,947,699.05","3,518,608,067.52",0.00,"1,850,576,356.39",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
1252,IRL,2021,USA,"365,750.32","116,789,098,743.36","47,722,215,781.01","2,618,854,524.20","487,319,372,485.76","178,253,685,695.79","57,343,185,519.50",25.15,"9,839,004,085.93",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
1463,KOR,2021,USA,"1,211,907.71","824,783,001,845.11","559,017,274,536.02","28,390,631,561.21","224,200,433,567.49","1,104,034,589,730.63","276,472,906,913.95",101.41,"74,519,757,742.42",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
1674,MUS,2021,USA,"19,369.61","46,500,991,834.26","11,604,211,467.10","50,262,556.25","12,821,548,483.00","48,045,833,879.20","3,751,209,344.20",5.02,"2,004,203,800.98",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07
1885,SWE,2021,USA,"558,567.16","237,341,756,278.42","90,105,424,295.56","8,273,613,657.98","69,309,325,418.42","320,202,274,336.05","81,370,857,431.80",85.22,"34,910,062,082.90",0.27,0.19,0.27,0.22,0.37,0.21,0.26,0.21,0.07


In [15]:
excluded_jurisdictions_dataset_2021 = excluded_jurisdictions_share_reported_2021.drop(columns=[
    'share_reported_total_profit_loss_by_partner',
    'share_reported_total_n_employees_by_partner',
    'share_reported_total_unrelated_party_revenues_by_partner',
    'share_reported_total_tangible_assets_except_cash_by_partner',
    'share_reported_total_payroll_by_partner',
    'share_reported_total_stated_capital_by_partner',
    'share_reported_total_total_revenues_by_partner',
    'share_reported_total_related_party_revenues_by_partner',
    'share_reported_total_holding_or_managing_ip_by_partner'
])

excluded_jurisdictions_dataset_2021[excluded_jurisdictions_dataset_2021['iso_partner'] == 'USA']

,iso_parent,year,iso_partner,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,profit_loss_before_income_tax_corrected
197,AUT,2021,USA,"395,200.37","198,384,558,241.92","88,382,568,513.57","8,279,413,048.09","55,739,216,511.75","243,376,521,454.14","47,859,082,489.78",38.80,"17,020,131,292.33"
408,CZE,2021,USA,"84,924.43","65,008,550,598.34","11,807,579,744.85","957,915,215.59","6,291,819,603.45","93,346,621,634.09","26,765,507,134.86",0.00,"1,566,599,401.38"
619,FIN,2021,USA,"188,177.98","223,367,474,028.89","40,650,649,182.00","3,732,261,724.25","126,077,473,165.25","289,216,920,972.42","66,399,822,747.38",14.52,"11,400,275,120.06"
830,GBR,2021,USA,"2,052,368.76","1,333,196,076,210.98","795,128,909,387.10","41,445,185,791.12","2,846,930,350,558.64","1,836,243,132,418.60","489,022,776,909.91",240.15,"162,785,567,779.30"
1041,HUN,2021,USA,"25,654.97","15,872,356,348.74","4,299,453,293.56","352,522,251.87","3,262,622,391.44","19,088,947,699.05","3,518,608,067.52",0.00,"1,850,576,356.39"
1252,IRL,2021,USA,"365,750.32","116,789,098,743.36","47,722,215,781.01","2,618,854,524.20","487,319,372,485.76","178,253,685,695.79","57,343,185,519.50",25.15,"9,839,004,085.93"
1463,KOR,2021,USA,"1,211,907.71","824,783,001,845.11","559,017,274,536.02","28,390,631,561.21","224,200,433,567.49","1,104,034,589,730.63","276,472,906,913.95",101.41,"74,519,757,742.42"
1674,MUS,2021,USA,"19,369.61","46,500,991,834.26","11,604,211,467.10","50,262,556.25","12,821,548,483.00","48,045,833,879.20","3,751,209,344.20",5.02,"2,004,203,800.98"
1885,SWE,2021,USA,"558,567.16","237,341,756,278.42","90,105,424,295.56","8,273,613,657.98","69,309,325,418.42","320,202,274,336.05","81,370,857,431.80",85.22,"34,910,062,082.90"


In [16]:
final_misalignment_2021 = cbcr_sample[cbcr_sample['year'] == 2021].copy()

# Concatenate excluded_jurisdictions_dataset_2021
final_misalignment_2021 = pd.concat([final_misalignment_2021, excluded_jurisdictions_dataset_2021])


final_misalignment_2021[final_misalignment_2021['iso_partner'] == 'USA']

,iso_parent,parent_jurisdiction,iso_partner,partner_jurisdiction,year,unrelated_party_revenues,profit_loss_before_income_tax,adjusted_profit_loss_before_income_tax,income_tax_paid_on_cash_basis,income_tax_accrued_current_year,n_employees,tangible_assets_except_cash,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip,n_cbcr,n_cbcr_groups,n_entities,profit_loss_before_income_tax_corrected,ln_profit_loss_before_income_tax_corrected,ln_unrelated_party_revenues,ln_n_employees,ln_tangible_assets_except_cash,ln_stated_capital,ln_total_revenues,ln_related_party_revenues,ln_holding_or_managing_ip,etr_domestic,etr_domestic_corrected,etr_foreign,etr_foreign_corrected,etr_average,etr_average_corrected,cit,gdp_current_usd,population,gdp,wage_monthly,payroll,ln_wage_monthly,ln_gdp_current_usd,ln_population,gvt_health_expenditure,ln_gvt_health_expenditure,tax_revenue_pct_gdp,tax_revenue_current_usd,cthi_2021_share,cthi_2021_score,region_tjn,ukt,gbr_oct,nld_oct,oecd_oct,oecd,eu
140,ARE,United Arab Emirates,USA,United States,2021,"22,510,101,353.00","-566,015,448.00",NaN,"33,594,827.00","41,196,000.00","28,384.00","11,019,650,222.00","33,844,332,143.00","25,838,686,878.00","3,328,585,524.00",19.00,28.00,28.00,438.00,"-566,015,448.00",0.00,23.84,10.25,23.12,24.25,23.98,21.93,3.00,0.14,0.18,0.16,0.16,0.14,0.18,0.27,"23,594,031,000,000.00","332,048,977.00",NaN,"4,600.13","1,566,840,057.22",8.43,30.79,19.62,"2,242,777,276,178.77",28.44,11.30,"2,666,221,799,678.12",0.01,46.90,Northern America,0.00,0.00,0.00,0.00,1.00,0.00
248,ARG,Argentina,USA,United States,2021,"113,406,031.00","67,155,125.00",NaN,"940,629.00","1,346,224.00",114.00,"103,940,740.00","1,762,118,722.00","132,596,801.00","19,190,770.00",2.00,16.00,16.00,38.00,"67,155,125.00",18.02,18.55,4.74,18.46,21.29,18.70,16.77,1.10,0.14,0.18,0.16,0.16,0.14,0.18,0.27,"23,594,031,000,000.00","332,048,977.00",NaN,"4,600.13","6,292,973.74",8.43,30.79,19.62,"2,242,777,276,178.77",28.44,11.30,"2,666,221,799,678.12",0.01,46.90,Northern America,0.00,0.00,0.00,0.00,1.00,0.00
720,AUS,Australia,USA,United States,2021,"57,209,834,217.00","8,095,396,499.00",NaN,"379,083,911.00","995,838,351.00","89,401.00","37,039,556,556.00","174,000,000,000.00","73,738,510,494.00","16,528,676,277.00",27.00,83.00,83.00,"1,198.00","8,095,396,499.00",22.81,24.77,11.40,24.34,25.88,25.02,23.53,3.33,0.14,0.18,0.16,0.16,0.14,0.18,0.27,"23,594,031,000,000.00","332,048,977.00",NaN,"4,600.13","4,935,071,447.12",8.43,30.79,19.62,"2,242,777,276,178.77",28.44,11.30,"2,666,221,799,678.12",0.01,46.90,Northern America,0.00,0.00,0.00,0.00,1.00,0.00
830,AZE,Azerbaijan,USA,United States,2021,"748,376,581.00","-5,934,134.00",NaN,"8,850.00","4,774.00",21.00,"7,712,734.00","20,001,000.00","751,937,979.00","3,561,398.00",NaN,2.00,2.00,3.00,"-5,934,134.00",0.00,20.43,3.09,15.86,16.81,20.44,15.09,NaN,0.14,0.18,0.16,0.16,0.14,0.18,0.27,"23,594,031,000,000.00","332,048,977.00",NaN,"4,600.13","1,159,232.00",8.43,30.79,19.62,"2,242,777,276,178.77",28.44,11.30,"2,666,221,799,678.12",0.01,46.90,Northern America,0.00,0.00,0.00,0.00,1.00,0.00
997,BEL,Belgium,USA,United States,2021,"43,812,500,000.00","2,376,700,000.00",NaN,"559,700,000.00","807,700,000.00","58,400.00","14,989,500,000.00","380,000,000,000.00","54,828,300,000.00","11,015,800,000.00",25.00,48.00,48.00,340.00,"2,376,700,000.00",21.59,24.50,10.98,23.43,26.66,24.73,23.12,3.26,0.14,0.18,0.16,0.16,0.14,0.18,0.27,"23,594,031,000,000.00","332,048,977.00",NaN,"4,600.13","3,223,769,001.60",8.43,30.79,19.62,"2,242,777,276,178.77",28.44,11.30,"2,666,221,799,678.12",0.01,46.90,Northern America,0.00,0.00,0.00,0.00,1.00,0.00
1060,BHR,Bahrain,USA,United States,2021,"31,352,002.00","28,628,903.00",NaN,"1,933,590.00","-7,006,209.00",46.00,"494,407.00","1,253,597.00","491,165,142.00","459,813,140.00",NaN,2.00,2.00,2.00,"28,628,903.00",17.17,17.26,3.85,13.11,14.04,20.01,19.95,NaN,0.14,0.18,0.16,0.16,0.14,0.18,0.27,"23,594,031,000,000.00","332,048,977.00",NaN,"4,600.13","2,539,270.10",8.4

In [17]:
# Initialize a list to store the aggregate results
results_sample = []

# Start the estimates
misalignment_final_estimates_2021 = final_misalignment_2021[final_misalignment_2021['year'] == 2021].copy()
misalignment_final_estimates_2021 = calculate_misalignment(misalignment_final_estimates_2021, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Perform the groupby operation on 'iso_partner'
country_results_2021 = misalignment_final_estimates_2021.groupby(['iso_partner']).agg(
    negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
    positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
    theoretical_profit=('theoretical_profit', 'sum'),
    reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
).reset_index()

# Convert results to millions
country_results_2021['negative_misalignment'] = -country_results_2021['negative_misalignment'] / 1e6
country_results_2021['positive_misalignment'] = country_results_2021['positive_misalignment'] / 1e6
country_results_2021['theoretical_profit'] = country_results_2021['theoretical_profit'] / 1e6
country_results_2021['reported_profit'] = country_results_2021['reported_profit'] / 1e6

# Merge the unique columns back into the result
country_results_2021 = country_results_2021.merge(unique_columns, on='iso_partner', how='left')

# Calculate other relevant variables
country_results_2021['tax_revenue_loss'] = country_results_2021['negative_misalignment'] * country_results_2021['cit']
country_results_2021['tax_revenue_gain'] = country_results_2021['positive_misalignment'] * country_results_2021['etr_average_corrected']

country_results_2021['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
    country_results_2021['gvt_health_expenditure'] == 0, 
    np.nan, 
    country_results_2021['tax_revenue_loss'] / (country_results_2021['gvt_health_expenditure'] / 1e6)
)
    
country_results_2021['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
    country_results_2021['tax_revenue_current_usd'] == 0, 
    np.nan, 
    country_results_2021['tax_revenue_loss'] / (country_results_2021['tax_revenue_current_usd'] / 1e6)
)

# Calculate totals
total_positive_misalignment = country_results_2021['positive_misalignment'].sum()
total_negative_misalignment = country_results_2021['negative_misalignment'].sum()
total_profits = country_results_2021['reported_profit'].sum()
misaligned_of_total_profits = total_positive_misalignment / total_profits
total_tax_revenue_loss = country_results_2021['tax_revenue_loss'].sum()
total_tax_revenue_gain = country_results_2021['tax_revenue_gain'].sum()
average_tax_revenue_loss_pct_of_gvt_health_expenditure = country_results_2021['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
average_tax_revenue_loss_pct_of_total_tax_revenues = country_results_2021['tax_revenue_loss_pct_of_total_tax_revenues'].mean()

print(f"Year {2021}: Positive Misalignment: {total_positive_misalignment}, Negative Misalignment: {total_negative_misalignment}, Shifted of total profits: {misaligned_of_total_profits}, "
        f"Total tax revenue loss: {total_tax_revenue_loss}, Total tax revenue gain: {total_tax_revenue_gain}")

# Calculate countries' fractions of totals
country_results_2021['tax_revenue_loss_caused_pct_of_total'] = country_results_2021['positive_misalignment'] / total_positive_misalignment
country_results_2021['tax_revenue_loss_caused_usd'] = country_results_2021['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss
country_results_2021['tax_revenue_loss_suffered_pct_of_total'] = country_results_2021['tax_revenue_loss'] / total_tax_revenue_loss

country_results_2021 = country_results_2021.sort_values(by='iso_partner')
country_results_2021.to_csv(f'{output_tables}/Scaling_Mario/SOTJ_sample_countries_2021_robustness_check.csv', index=False) # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE
    
# Append aggregate results to the list
results_sample.append({
    'year': 2021,
    'total_positive_misalignment': total_positive_misalignment,
    'total_negative_misalignment': total_negative_misalignment,
    'total_profits': total_profits,
    'misaligned_of_total_profits': misaligned_of_total_profits,
    'total_tax_revenue_loss': total_tax_revenue_loss,
    'total_tax_revenue_gain': total_tax_revenue_gain,
    'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_tax_revenue_loss_pct_of_gvt_health_expenditure,
    'average_tax_revenue_loss_pct_of_total_tax_revenues': average_tax_revenue_loss_pct_of_total_tax_revenues
})

# Convert aggregate results to a DataFrame
results_sample_df = pd.DataFrame(results_sample)

# Save the aggregated results to a CSV or Excel file
results_sample_df.to_csv(f'{output_tables}/Scaling_Mario/SOTJ_sample_aggregate_results_2021_robustness_check.csv', index=False)  # CHANGE FILE NAME HERE, DEPENDING ON FORMULA USE


Year 2021: Positive Misalignment: 1420180.0438085557, Negative Misalignment: 1420180.0438085552, Shifted of total profits: 0.15895261084432838, Total tax revenue loss: 380405.95436632115, Total tax revenue gain: 118593.46762131003


/var/folders/kj/y_rfkb696353czt_6gnx_jkc0000gn/T/ipykernel_23015/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


In [18]:
# Open the CSV file f'{output_tables}/Scaling_Mario/SOTJ_sample_2016.csv'
sotj_2021_robustness_check = pd.read_csv(f'{output_tables}/Scaling_Mario/SOTJ_sample_countries_2021_robustness_check.csv')

sotj_2021_robustness_check

,iso_partner,negative_misalignment,positive_misalignment,theoretical_profit,reported_profit,partner_jurisdiction,etr_average_corrected,cit,tax_revenue_current_usd,gvt_health_expenditure,region_tjn,ukt,oecd,oecd_oct,nld_oct,tax_revenue_loss,tax_revenue_gain,tax_revenue_loss_pct_of_gvt_health_expenditure,tax_revenue_loss_pct_of_total_tax_revenues,tax_revenue_loss_caused_pct_of_total,tax_revenue_loss_caused_usd,tax_revenue_loss_suffered_pct_of_total
0,ABW,68.57,0.00,66.35,-10.55,Aruba,0.24,0.25,NaN,NaN,Caribbean/American isl.,0.00,0.00,1.00,1.00,17.14,0.00,NaN,NaN,0.00,0.00,0.00
1,AFG,86.43,3.07,101.69,-38.12,Afghanistan,0.07,0.20,NaN,"107,697,735.61",Asia,0.00,0.00,0.00,0.00,17.29,0.22,0.16,NaN,0.00,0.82,0.00
2,AGO,168.28,329.58,"5,191.90","3,353.05",Angola,0.36,0.30,NaN,"1,279,772,420.68",Africa,0.00,0.00,0.00,0.00,50.48,120.01,0.04,NaN,0.00,88.28,0.00
3,AIA,1.82,0.00,0.00,-3.13,Anguilla,0.00,NaN,NaN,NaN,Caribbean/American isl.,1.00,0.00,1.00,0.00,NaN,0.00,NaN,NaN,0.00,0.00,NaN
4,ALB,44.82,4.30,78.66,27.29,Albania,0.08,0.15,"3,263,109,930.25","526,532,522.89",Europe,0.00,0.00,0.00,0.00,6.72,0.36,0.01,0.00,0.00,1.15,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,XKV,5.04,3.39,54.48,34.05,Kosovo,0.08,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,NaN,0.27,NaN,NaN,0.00,0.91,NaN
207,YEM,165.19,2.03,490.73,16.45,Yemen,0.45,0.20,NaN,NaN,Asia,0.00,0.00,0.00,0.00,33.04,0.91,NaN,NaN,0.00,0.54,0.00
208,ZAF,"4,556.62","18,883.48","42,481.38","55,421.65",South Africa,0.14,0.28,"108,987,088,441.14","20,932,576,170.69",Africa,0.00,0.00,0.00,0.00,"1,275.85","2,573.69",0.06,0.01,0.01,"5,058.08",0.00
209,ZMB,156.27,0.00,541.97,343.88,Zambia,0.26,0.35,"3,707,869,506.76","623,797,186.91",Africa,0.00,0.00,0.00,0.00,54.70,0.00,0.09,0.01,0.00,0.00,0.00


In [19]:
sotj_aggregate_robustness_check = pd.read_csv(f'{output_tables}/Scaling_Mario/SOTJ_sample_aggregate_results_2021_robustness_check.csv')
sotj_aggregate_robustness_check


,year,total_positive_misalignment,total_negative_misalignment,total_profits,misaligned_of_total_profits,total_tax_revenue_loss,total_tax_revenue_gain,average_tax_revenue_loss_pct_of_gvt_health_expenditure,average_tax_revenue_loss_pct_of_total_tax_revenues
0,2021,"1,420,180.04","1,420,180.04","8,934,612.88",0.16,"380,405.95","118,593.47",0.61,0.04


38 countries decreased their cit. 
The averagre decrease is -0.05891280210526315
13 are from the OECD


16 countries increased their cit
The average increase is 0.036201477499999996
5 are from the OECD

Average decrease for all these countries
-0.030730793333333332

In [24]:
# Sort by iso_partner
comparison_cits = comparison_cits.sort_values(by=['iso_partner'])

# Drop duplicates
comparison_cits = comparison_cits.drop_duplicates()

# Rename cit as cit_2021
comparison_cits = comparison_cits.rename(columns={'cit': 'cit_2021'})

# Drop if cit or cit_2016 is NaN
comparison_cits = comparison_cits.dropna(subset=['cit_2021', 'cit_2016'])

# Order columns as: iso_partner, cit_2021, cit_2016
#comparison_cits = comparison_cits[['iso_partner', 'cit_2016', 'cit_2021']]

# Generate decrease in CIT by subtracting cit_2016 from cit
comparison_cits['decrease_cit'] = comparison_cits['cit_2021'] - comparison_cits['cit_2016']

comparison_cits
# Show me if decrease_cit is negative
decrease_cit = comparison_cits[comparison_cits['decrease_cit'] < 0]
increase_cit = comparison_cits[comparison_cits['decrease_cit'] > 0]

increase_cit
# Average decrease in CIT
#decrease_cit['decrease_cit'].mean()
# Average increase in CIT
#increase_cit['decrease_cit'].mean()

#Countries where it changed: concatenate decrease_cit and increase_cit
#countries_changed_cit = pd.concat([decrease_cit, increase_cit])

#countries_changed_cit['decrease_cit'].mean()


,iso_partner,cit_2021,oecd,cit_2016,decrease_cit
1587,BGD,0.33,0.00,0.25,0.08
1289,CHL,0.27,1.00,0.24,0.03
1201,DEU,0.30,1.00,0.30,0.00
951,ECU,0.25,0.00,0.22,0.03
2580,KOR,0.28,1.00,0.24,0.03
2774,LBN,0.17,0.00,0.15,0.02
571,LVA,0.20,1.00,0.15,0.05
348,OMN,0.15,0.00,0.12,0.03
2800,PER,0.29,0.00,0.28,0.01
1346,PRT,0.32,1.00,0.29,0.02
